# IndetBench · p(UNKNOWN) su modelli più grandi (Colab, A100-80GB)

Versione del notebook: **N-notebook-4** (15 novembre 2026). Benchmark e codice di calcolo vengono dalla repository `Sva76/indet-bench` al commit di main `2aa1f62`. Il calcolo di p(UNKNOWN), W, soglia e report è identico a quello del commit `c086e65` usato per i modelli nuovi del 14 novembre.

**Che cosa fa.** Stesso protocollo del notebook dei modelli nuovi (N-notebook-3): le 800 coppie B/C congelate del Test U2 (set PN), le stesse impostazioni (bfloat16, eager, batch 1, greedy, 8 token) e lo stesso `report` con la soglia di competenza vincolante, le baseline descrittive, i tre livelli e le anomalie DA CLASSIFICARE. Cambiano soltanto i modelli, scelti per la **scala a famiglia fissa**: Qwen3-32B, OLMo-2-32B, Gemma-2-27B e Llama-3.3-70B. Quest'ultimo è **quantizzato a 4 bit** (nf4), perché in bfloat16 non entra nella GPU: l'eccezione è dichiarata nel manifest e nel rapporto.

**Novità rispetto alla v3**
- Controllo di copertura delle etichette, introdotto dopo il caso Yi. Su 24 righe il notebook misura la massa al primo token sulle etichette e il token più probabile, prima dell'inferenza completa. Se la massa mediana è < 0.5, il modello è NON MISURABILE; lo stesso controllo si ripete sull'insieme completo.
- Controllo della memoria della GPU: i modelli da 27–33B richiedono la A100 da 80 GB. Su una GPU troppo piccola il modello viene saltato con un errore, non con un esito, così si può rilanciare.
- `GH_TOKEN` facoltativo: la repository è pubblica.

**Come leggere i risultati.** Tutti i risultati sono **ESPLORATIVI**. Il controllo positivo è sempre Qwen3-8B, che deve riprodurre il Test U2. Il confronto con le baseline è descrittivo: non si può dire che p(UNKNOWN) sia superiore alle baseline.

**Come partire.**
1. Runtime → **A100**, poi controlla con `!nvidia-smi` che sia la **A100-SXM4-80GB**.
2. Secrets: `HF_TOKEN` con accesso al notebook. Accetta le licenze di Gemma-2-27B e di Llama-3.3-70B su Hugging Face. `testU2_risultati.zip` deve essere in MyDrive (controllo CPU, cella 4b).
3. **Prova**: `MODALITA = "prova"` con `qwen_controllo` + `gemma2_27b`, poi `AVVIA_GPU = True`.
4. **Completo**: `MODALITA = "completo"` e `MODELLI = ["qwen_controllo", "qwen3_32b", "olmo2_32b", "gemma2_27b", "llama33_70b_4bit"]`. Il download è di circa 60 GB per ciascun modello da 27–33B e di circa 140 GB per il 70B: lo spazio disco di Colab va controllato.

Le uscite vanno in `MyDrive/indetbench_modelli_grandi/<modalità>/`, una sola copia, con la stessa struttura del notebook dei modelli nuovi. Se la sessione cade, rilancia: riprende dai blocchi salvati.

In [ ]:
# 1. Configurazione. Le celle 2–4 girano anche su CPU (preflight); l'inferenza parte solo con AVVIA_GPU = True.
MODALITA = "prova"          # "prova" (9 storie, pochi minuti) oppure "completo" (200 storie, 800 coppie)
MODELLI = ["qwen_controllo", "gemma2_27b"]
# completo: ["qwen_controllo", "qwen3_32b", "olmo2_32b", "gemma2_27b", "llama33_70b_4bit"]
# Serve la sessione A100-SXM4-80GB: i modelli da 27–33B in bfloat16 non entrano nella 40 GB.
AVVIA_GPU = False
SOLO_A100 = True            # ferma l'inferenza su GPU diverse da A100 (come nel Test M)
USA_DRIVE = True            # uscite in MyDrive/indetbench_modelli_grandi/<modalità>/ (una sola copia)

# Codice e benchmark: repository privata, commit fissato (PR #3, soglia di competenza compresa).
REPO = "Sva76/indet-bench"
REPO_COMMIT = "2aa1f6280f14d2c85c02923f9fce6e425678a293"   # main del 15 nov; calcolo identico a c086e65
# Controllo su CPU dei 4 W del Test U2 (cella 4b), obbligatorio prima del completo: ZIP archiviato del Test U2
# (contiene punteggi.csv). Una sola copia in MyDrive; percorso relativo a /content/drive/MyDrive.
ARCHIVIO_TEST_U2 = "testU2_risultati.zip"
# Secrets di Colab (icona a chiave), con «Accesso al notebook» attivo per QUESTO notebook:
#   GH_TOKEN: facoltativo (la repository ora è pubblica); se presente, usato in sola lettura
#   HF_TOKEN: token Hugging Face (Gemma-2-27B e Llama-3.3-70B sono gated: accetta prima le licenze)

In [ ]:
# 2. Drive, repository al commit fissato, benchmark congelato. Nessuna GPU richiesta.
import hashlib, json, os, re, subprocess, sys, zipfile
from pathlib import Path

VERSIONE = "N-notebook-4"
assert MODALITA in ("prova", "completo"), MODALITA
SMOKE = MODALITA == "prova"
REPO_DIR = Path(os.environ.get("INDETBENCH_REPO_DIR", "/content/indet-bench"))

_SEGRETI = []

def nascondi(testo):
    """Toglie i token noti da qualsiasi testo che finisce a schermo o nei file."""
    testo = str(testo)
    for s in _SEGRETI:
        if s:
            testo = testo.replace(s, "***")
    return testo

def segreto(nome):
    valore = os.environ.get(nome)
    if not valore:
        try:
            from google.colab import userdata
            valore = userdata.get(nome)
        except Exception:
            valore = None
    if valore:
        _SEGRETI.append(valore)
    return valore

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} fallito: {r.stderr.strip()[:300]}")
    return r.stdout.strip()

if not (REPO_DIR / ".git").exists():
    gh = segreto("GH_TOKEN")    # facoltativo: la repository è pubblica
    import base64
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("init", "-q", str(REPO_DIR), cwd=REPO_DIR.parent)
    # Il token passa solo in una variabile d'ambiente del processo git (intestazione HTTP): non compare
    # negli argomenti, nell'URL, in .git/config o in FETCH_HEAD, e non viene stampato.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    cred = None
    if gh:
        cred = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
        _SEGRETI.append(cred)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {cred}")
    r = subprocess.run(["git", "fetch", "-q", "--depth", "1", f"https://github.com/{REPO}.git", REPO_COMMIT],
                       cwd=REPO_DIR, capture_output=True, text=True, env=env)
    del env, cred
    if r.returncode:
        raise RuntimeError("Download della repository fallito: " + nascondi(r.stderr)[:300])
    git("checkout", "-q", "FETCH_HEAD")
    (REPO_DIR / ".git" / "FETCH_HEAD").unlink(missing_ok=True)
assert git("rev-parse", "HEAD") == REPO_COMMIT, "La copia locale non è al commit fissato"
assert not git("status", "--porcelain", "--untracked-files=no"), "La copia locale della repository è modificata"
sys.path.insert(0, str(REPO_DIR / "src"))

from indetbench import original_benchmark as ob, hf_runner as hr
from indetbench.metrics import PairScore, w_score

FROZEN = ob.load_frozen(REPO_DIR / "research_originals/step2/frozen/M_Original_4800_righe.json.gz")  # controlla SHA-256
TUTTE = ob.export_rows(FROZEN, "PN")          # 1600 righe B/C, 800 coppie, 200 storie, prompt invariati
GOLD = {r["row_id"]: r for r in FROZEN}

def storie_prova():
    # 3 storie per famiglia, le prime in ordine; comprendono V2001–V2003 del controllo Qwen
    scelte = {}
    for r in TUTTE:
        lista = scelte.setdefault(r["family"], [])
        if r["block_id"] not in lista and len(lista) < 3:
            lista.append(r["block_id"])
    return {s for v in scelte.values() for s in v}

RIGHE = [r for r in TUTTE if r["block_id"] in storie_prova()] if SMOKE else TUTTE
DATA_SHA = hashlib.sha256("\n".join(r["row_id"] + "\t" + r["prompt"] for r in RIGHE).encode()).hexdigest()
n_coppie = len({r["pair_id"] for r in RIGHE})
assert SMOKE or (len(RIGHE) == 1600 and n_coppie == 800)

ROOT = Path("/content/drive/MyDrive" if USA_DRIVE else "/content") / "indetbench_modelli_grandi" / MODALITA
if USA_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT.mkdir(parents=True, exist_ok=True)
print(f"Modalità {MODALITA} | repository {REPO_COMMIT[:7]} | benchmark SHA-256 {ob.FROZEN_SHA256[:12]} (verificato)")
print(f"Righe {len(RIGHE)} | coppie {n_coppie} | storie {len({r['block_id'] for r in RIGHE})} | dati {DATA_SHA[:16]}")
print(f"Uscite in {ROOT}")
esempio = next(r for r in RIGHE if r["side"] == "C")
print(f"\n--- esempio C ({esempio['pair_id']}) ---\n{esempio['prompt']}")

In [ ]:
# 3. Modelli e revisioni, proposti il 15 novembre 2026 prima di qualsiasi dato sui modelli grandi (da confermare da Simona).
# Le revisioni sono commit completi di Hugging Face; la cella 5 controlla che esistano prima di scaricare i pesi.
# Impostazioni identiche al Test U2: bfloat16, attenzione eager, batch 1, greedy, 8 token nuovi, solo messaggio utente.
# Unica eccezione dichiarata: Llama-3.3-70B in 4 bit (nf4, bitsandbytes, calcolo in bfloat16), perché in bf16 (~141 GB) non entra.
MODEL_CONFIG = {
    "qwen_controllo": dict(id="Qwen/Qwen3-8B", sha="b968826d9c46dd6066d109eabc6255188de91218", key="qwen",
                           ruolo="controllo positivo (modello originale del Test U2)", licenza="Apache-2.0",
                           quant=None, gpu_min_gb=20, disco_gb=17, famiglia_di="Qwen3-8B"),
    "qwen3_32b": dict(id="Qwen/Qwen3-32B", sha="9216db5781bf21249d130ec9da846c4624c16137", key="qwen",
                      ruolo="modello grande, stessa famiglia di Qwen3-8B", licenza="Apache-2.0",
                      quant=None, gpu_min_gb=75, disco_gb=67, famiglia_di="Qwen3-8B"),
    "olmo2_32b": dict(id="allenai/OLMo-2-0325-32B-Instruct", sha="b96024342a77a69aa0dda815c3454a671f477463", key="other",
                      ruolo="modello grande, stessa famiglia di OLMo-2-7B", licenza="Apache-2.0",
                      quant=None, gpu_min_gb=75, disco_gb=66, famiglia_di="OLMo-2-1124-7B-Instruct"),
    "gemma2_27b": dict(id="google/gemma-2-27b-it", sha="aaf20e6b9f4c0fcf043f6fb2a2068419086d77b0", key="other",
                       ruolo="modello grande, stessa famiglia di Gemma-2-9B", licenza="Gemma Terms of Use (gated)",
                       quant=None, gpu_min_gb=75, disco_gb=56, famiglia_di="gemma-2-9b-it"),
    "llama33_70b_4bit": dict(id="meta-llama/Llama-3.3-70B-Instruct", sha="6f6073b423013f6a7d4d9f39144961bfbfbc386b",
                             key="other", ruolo="modello grande QUANTIZZATO a 4 bit, stessa famiglia di Llama-3.1-8B",
                             licenza="Llama 3.3 Community License (gated)", quant="nf4", gpu_min_gb=45, disco_gb=143,
                             famiglia_di="Llama-3.1-8B-Instruct"),
}
# Regola tecnica fissata il 15 nov prima dei dati (dopo il caso Yi del 14 nov): se la massa mediana di
# YES+NO+UNKNOWN al primo token è < 0.5, le etichette misurate non coprono il token che il modello genera e
# p(UNKNOWN) non è la quantità del protocollo: il modello è NON MISURABILE (motivo tecnico), qualunque sia W.
MASSA_MEDIANA_MIN = 0.5
for k in MODELLI:
    assert k in MODEL_CONFIG, f"Modello sconosciuto: {k}"
    hr.validate_revision(MODEL_CONFIG[k]["sha"])

# Controllo positivo: 24 valori di p(UNKNOWN) di Qwen3-8B sulle storie V2001–V2003 (set PN, B e C),
# dal punteggi.csv archiviato del Test U2 (9 nov 2026). Regola del Test U2: scarto massimo ≤ 1e-3.
REF_QWEN_PN = [["PN|V2001|POS|prima|granted|B", 1.0816490064735618e-11, "NO"], ["PN|V2001|POS|ultima|granted|B", 1.3711326118936995e-06, "NO"], ["PN|V2001|POS|prima|granted|C", 5.905622102453376e-10, "YES"], ["PN|V2001|POS|ultima|granted|C", 0.9859082102775574, "UNKNOWN"], ["PN|V2001|NOT|prima|granted|B", 1.955604211900664e-08, "NO"], ["PN|V2001|NOT|ultima|granted|B", 0.0001234118099091, "NO"], ["PN|V2001|NOT|prima|granted|C", 6.826222431755016e-08, "YES"], ["PN|V2001|NOT|ultima|granted|C", 0.5621962547302246, "UNKNOWN"], ["PN|V2002|POS|prima|granted|B", 3.266230601955966e-13, "YES"], ["PN|V2002|POS|ultima|granted|B", 1.5428827106981485e-13, "YES"], ["PN|V2002|POS|prima|granted|C", 0.0116246016696095, "NO"], ["PN|V2002|POS|ultima|granted|C", 0.0002034515491686, "YES"], ["PN|V2002|NOT|prima|granted|B", 6.826094534062578e-08, "YES"], ["PN|V2002|NOT|ultima|granted|B", 1.5231051975206356e-08, "YES"], ["PN|V2002|NOT|prima|granted|C", 2.1415366063592955e-05, "NO"], ["PN|V2002|NOT|ultima|granted|C", 0.0008244285709224, "YES"], ["PN|V2003|POS|prima|granted|B", 1.1862326232403577e-08, "YES"], ["PN|V2003|POS|ultima|granted|B", 3.059368793856265e-07, "YES"], ["PN|V2003|POS|prima|granted|C", 0.0001233841758221, "YES"], ["PN|V2003|POS|ultima|granted|C", 0.676827073097229, "UNKNOWN"], ["PN|V2003|NOT|prima|granted|B", 1.0130904229299633e-05, "YES"], ["PN|V2003|NOT|ultima|granted|B", 0.0004306302289478, "YES"], ["PN|V2003|NOT|prima|granted|C", 0.0005002911202609, "YES"], ["PN|V2003|NOT|ultima|granted|C", 0.1642791032791137, "YES"]]
# Controllo CPU (cella 4b): punteggi.csv del Test U2 (SHA-256 sotto) e W attesi su PN, 800 coppie.
PUNTEGGI_U2_SHA256 = "9701dce0dc7516665bc4a4c52026e52a0f9b7a2ccc3df01b660be7d9406d31aa"
W_ATTESI_U2_PN = {"qwen": 0.975, "mistral": 0.7825, "olmo": 0.69375, "llama": 0.74625}
NOMI_U2 = {"qwen": "Qwen/Qwen3-8B", "mistral": "mistralai/Mistral-Nemo-Instruct-2407",
           "olmo": "allenai/OLMo-2-1124-7B-Instruct", "llama": "meta-llama/Llama-3.1-8B-Instruct"}
W_QWEN_PN_TEST_U2 = 0.975          # Test U2, P−S, 800 coppie; nel completo il W deve coincidere entro 0.005
TOLL_VALORI, TOLL_W = 1e-3, 0.005

# Regole di lettura, fissate prima dei dati (vedi la nota che accompagna il notebook).
COPERTURA_MIN = 760                # coppie valide minime (come nel Test M); sotto: solo descrittivo
MASSA_BASSA, QUOTA_MASSA = 0.90, 0.05   # cautela se più del 5% delle righe ha massa YES+NO+UNKNOWN < 0.90 (Test U2)

In [ ]:
# 4. Analisi (solo CPU). Il risultato principale e la soglia di competenza vengono da indetbench.original_benchmark.report
# al commit fissato; qui si aggiungono le baseline dalle uscite e i controlli sugli artefatti, tutti DESCRITTIVI.
import math
from collections import Counter, defaultdict

SEED_BOOT = 20261108
LABELS = ("YES", "NO", "UNKNOWN")

def encode(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")

def fingerprint(obj):
    return hashlib.sha256(encode(obj)).hexdigest()

def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(encode(obj))
    os.replace(tmp, path)

def normalizzate(r):
    p = [float(r[f"p_{k.lower()}"]) for k in LABELS]
    s = max(sum(p), 1e-300)
    return [x / s for x in p]

NORMALIZZATA = "p_UNKNOWN normalizzata su YES+NO+UNKNOWN"
NOTA_NORMALIZZATA = ("La baseline «p(UNKNOWN) normalizzata» è p(UNKNOWN)/(p(YES)+p(NO)+p(UNKNOWN)): quando la massa "
                     "sulle tre risposte è circa 1 coincide con p(UNKNOWN), quindi la parità con essa è attesa e non è "
                     "un'anomalia del metodo. Resta nel confronto per trasparenza: se il suo W differisce da quello di "
                     "p(UNKNOWN), la differenza misura l'effetto della massa fuori dalle tre risposte. Non entra nella "
                     "segnalazione «una baseline eguaglia o supera p(UNKNOWN)», che usa solo le baseline indipendenti.")

MISURE = {   # punteggio più alto = più indeterminato
    "p_UNKNOWN (principale)": lambda r: float(r["score"]),
    NORMALIZZATA: lambda r: normalizzate(r)[2],
    "meno p massima normalizzata": lambda r: -max(normalizzate(r)),
    "entropia normalizzata": lambda r: -sum(x * math.log(max(x, 1e-300)) for x in normalizzate(r)),
    "risposta generata UNKNOWN (0/1)": lambda r: float(r["parsed"] == "UNKNOWN"),
    "lunghezza del prompt in token": lambda r: float(r["prompt_tokens"]),
}

def stima_W(rows, misura, boot, coppie_ok=None):
    pairs = [PairScore(r["pair_id"], r["block_id"], r["side"], misura(r)) for r in rows
             if coppie_ok is None or r["pair_id"] in coppie_ok]
    if not pairs:
        return dict(stima=None, ci95=None, coppie=0, storie=0)
    e = w_score(pairs, n_bootstrap=boot, seed=SEED_BOOT)
    ci = None if e.ci_lower is None else [e.ci_lower, e.ci_upper]
    return dict(stima=e.value, ci95=ci, coppie=e.n_pairs, storie=e.n_blocks)

def auroc(rows, misura):
    c = [misura(r) for r in rows if r["side"] == "C"]
    b = [misura(r) for r in rows if r["side"] == "B"]
    if not c or not b:
        return None
    vinte = sum((x > y) + 0.5 * (x == y) for x in c for y in b)
    return vinte / (len(c) * len(b))

def filtro_lunghezza(rows, lunghezze):
    # Regola del Test U2: B e C della stessa coppia devono avere lo stesso numero di token, altrimenti la coppia esce.
    per_coppia = defaultdict(set)
    for r in rows:
        per_coppia[r["pair_id"]].add(lunghezze[r["row_id"]])
    via = {p for p, s in per_coppia.items() if len(s) != 1}
    tenute = [r for r in rows if r["pair_id"] not in via]
    scarti = [dict(row_id=r["row_id"], pair_id=r["pair_id"], prompt_tokens=lunghezze[r["row_id"]])
              for r in rows if r["pair_id"] in via]
    return tenute, scarti

ATTRIBUTI = ("family", "variant", "position", "condition")

def distorsione_esclusioni(tutte, scarti):
    """Le coppie escluse sono distribuite come tutte le coppie? Tabella per attributo, solo descrittiva."""
    escluse = {s["pair_id"] for s in scarti}
    coppie = {r["pair_id"]: GOLD[r["row_id"]] for r in tutte if r["side"] == "B"}
    tab = {}
    for a in ATTRIBUTI:
        conta = defaultdict(lambda: [0, 0])
        for p, g in coppie.items():
            conta[g[a]][0] += 1
            conta[g[a]][1] += p in escluse
        tab[a if a != "condition" else "gold_B"] = {k: dict(coppie=n, escluse=e, quota=e / n) for k, (n, e) in sorted(conta.items())}
    storie = Counter(coppie[p]["story_id"] for p in escluse)
    quota = len(escluse) / max(len(coppie), 1)
    # concentrazione: un gruppo con almeno 5 escluse e quota più che doppia di quella complessiva
    concentrate = [f"{a}={k} ({v['escluse']}/{v['coppie']})" for a, d in tab.items() for k, v in d.items()
                   if v["escluse"] >= 5 and v["quota"] > 2 * quota]
    return dict(coppie_escluse=len(escluse), quota=quota, per_attributo=tab,
                storie_con_esclusioni=len(storie), max_per_storia=max(storie.values(), default=0),
                gruppi_concentrati=concentrate, elenco=sorted(escluse))

def analizza(nome, cfg, rows, scarti, smoke, extra=None, tutte=None):
    """rows: coppie tenute (analisi principale); tutte: tutte le righe misurate, comprese le coppie escluse."""
    boot = 0 if smoke else 2000
    rep = ob.report(rows, FROZEN, n_bootstrap=boot, seed=SEED_BOOT)
    coppie = rep["W_p_unknown"]["n_pairs"]
    if smoke:
        esito = "PROVA TECNICA (nessuna lettura scientifica)"
    elif coppie < COPERTURA_MIN:
        esito = f"DESCRITTIVO (solo {coppie} coppie valide, minimo {COPERTURA_MIN})"
    else:
        esito = rep["verdict"]
    per_coppia = defaultdict(dict)
    for r in rows:
        per_coppia[r["pair_id"]][r["side"]] = r
    attr = {p: GOLD[d["B"]["row_id"]] for p, d in per_coppia.items()}
    def gruppo(f):
        out = defaultdict(set)
        for p, g in attr.items():
            out[f(p, g)].add(p)
        return out
    controlli = {}
    for nome_g, f in (("famiglia", lambda p, g: g["family"]), ("variante", lambda p, g: g["variant"]),
                      ("posizione", lambda p, g: g["position"]),
                      ("famiglia x variante", lambda p, g: f"{g['family']}/{g['variant']}")):
        controlli[f"W per {nome_g}"] = {k: stima_W(rows, MISURE["p_UNKNOWN (principale)"], boot, v)
                                        for k, v in sorted(gruppo(f).items())}
    unk = {p: (d["B"]["parsed"] == "UNKNOWN", d["C"]["parsed"] == "UNKNOWN") for p, d in per_coppia.items()}
    parita = {"nessuna delle due risposte UNKNOWN": {p for p, u in unk.items() if not any(u)},
              "entrambe le risposte UNKNOWN": {p for p, u in unk.items() if all(u)},
              "C senza risposta UNKNOWN": {p for p, u in unk.items() if not u[1]}}
    controlli["W a parità di risposta generata"] = {k: stima_W(rows, MISURE["p_UNKNOWN (principale)"], boot, v)
                                                    for k, v in parita.items()}
    risposte = {s: Counter(r["parsed"] for r in rows if r["side"] == s) for s in ("B", "C")}
    massa_bassa = sum(float(r["mass_total"]) < MASSA_BASSA for r in rows) / max(len(rows), 1)
    return dict(
        modello=cfg["id"], revisione=cfg["sha"], ruolo=cfg["ruolo"], chiave=nome,
        classe=("CONTROLLO POSITIVO" if nome == "qwen_controllo" else
                "ESPLORATIVO: modello nuovo, non è una validazione indipendente"),
        esito=esito, report_indetbench=rep,
        baseline_dalle_uscite={k: dict(W=stima_W(rows, m, boot), AUROC=auroc(rows, m)) for k, m in MISURE.items()},
        controlli_artefatti=controlli,
        risposte_generate={s: dict(c) for s, c in risposte.items()},
        quota_UNKNOWN={s: risposte[s]["UNKNOWN"] / max(sum(risposte[s].values()), 1) for s in ("B", "C")},
        coppie_scartate_lunghezza=len({s["pair_id"] for s in scarti}),
        esclusioni=dict(
            distorsione=distorsione_esclusioni(tutte or rows, scarti),
            # sensibilità: stesso W calcolato anche sulle coppie escluse (misurate comunque); solo descrittivo
            W_sensibilita_tutte_le_coppie=(stima_W(tutte, MISURE["p_UNKNOWN (principale)"], boot)
                                           if tutte and scarti else None)),
        quota_massa_bassa=massa_bassa, cautela_massa=massa_bassa > QUOTA_MASSA,
        **(extra or {}))


def riga_riepilogo(e):
    rep = e["report_indetbench"]; g = rep["competence_gate"]; w = rep["W_p_unknown"]
    base = e["baseline_dalle_uscite"]
    fam = rep["W_by_family_descriptive"]
    return dict(modello=e["chiave"], classe=e["classe"].split(":")[0], esito=e["esito"],
                W=w["value"], IC95_basso=w["ci_lower"], IC95_alto=w["ci_upper"], coppie=w["n_pairs"],
                acc_B=g["accuracy_B"], acc_B_gold_NO=g["accuracy_B_by_gold"]["NO"],
                acc_B_gold_YES=g["accuracy_B_by_gold"]["YES"], soglia_superata=g["passed"],
                W_AND=fam["AND"], W_OR=fam["OR"], W_IMPLIES=fam["IMPLIES"],
                W_entropia=base["entropia normalizzata"]["W"]["stima"],
                W_meno_p_max=base["meno p massima normalizzata"]["W"]["stima"],
                W_risposta_UNKNOWN=base["risposta generata UNKNOWN (0/1)"]["W"]["stima"],
                W_lunghezza=base["lunghezza del prompt in token"]["W"]["stima"],
                UNK_B=e["quota_UNKNOWN"]["B"], UNK_C=e["quota_UNKNOWN"]["C"],
                scarti_lunghezza=e["coppie_scartate_lunghezza"], cautela_massa=e["cautela_massa"])

def _segnala(lista, cosa, prova, categoria_possibile):
    # Ogni anomalia resta «DA CLASSIFICARE»: la causa va documentata da chi legge, con prove e controlli.
    lista.append(dict(anomalia=cosa, prova=prova, categorie_possibili=categoria_possibile,
                      classificazione="DA CLASSIFICARE", causa_ipotizzata="", controlli_effettuati="",
                      cosa_resta_incerto=""))

def livelli(e):
    """Tre livelli separati (criterio di Simona, 14 nov 2026) e anomalie da classificare."""
    rep = e["report_indetbench"]; g = rep["competence_gate"]; w = rep["W_p_unknown"]
    base = e["baseline_dalle_uscite"]; ctrl = e["controlli_artefatti"]
    anomalie = []
    risposte = Counter(); [risposte.update(c) for c in e["risposte_generate"].values()]
    n = sum(risposte.values())
    if e["coppie_scartate_lunghezza"]:
        _segnala(anomalie, "coppie con B e C di lunghezza diversa in token, escluse",
                 f"{e['coppie_scartate_lunghezza']} coppie", "problema tecnico (tokenizzazione) / limite del protocollo")
    dist = e["esclusioni"]["distorsione"]
    if dist["gruppi_concentrati"]:
        _segnala(anomalie, "esclusioni per lunghezza concentrate in alcuni gruppi (possibile distorsione)",
                 "; ".join(dist["gruppi_concentrati"]), "limite del protocollo / problema tecnico (tokenizzazione)")
    sens = e["esclusioni"]["W_sensibilita_tutte_le_coppie"]
    if sens and w["value"] is not None and abs(sens["stima"] - w["value"]) >= 0.02:
        _segnala(anomalie, "W cambia di almeno 0.02 includendo le coppie escluse",
                 f"{w['value']:.4f} senza, {sens['stima']:.4f} con", "limite del protocollo")
    if risposte["ALTRO"] / max(n, 1) > 0.05:
        _segnala(anomalie, "risposte non leggibili come YES/NO/UNKNOWN", f"{risposte['ALTRO']}/{n}",
                 "comportamento del modello / problema tecnico (template, lettura della risposta)")
    if e["cautela_massa"]:
        _segnala(anomalie, "massa di probabilità YES+NO+UNKNOWN bassa al primo token",
                 f"{e['quota_massa_bassa']:.1%} delle righe < {MASSA_BASSA}",
                 "comportamento del modello (formato della risposta) / problema tecnico (template)")
    for w_ in rep["warnings"]:
        _segnala(anomalie, w_, "report di indetbench", "comportamento del modello")
    if not g["passed"]:
        _segnala(anomalie, "soglia di competenza sui B non superata", f"parti fallite: {g['failed_parts']}",
                 "comportamento del modello / limite del protocollo")
    fam = [v for v in rep["W_by_family_descriptive"].values() if v is not None]
    if fam and max(fam) - min(fam) >= 0.20:
        _segnala(anomalie, "W molto diverso fra famiglie (differenza ≥ 0.20, soglia descrittiva del Test M)",
                 json.dumps(rep["W_by_family_descriptive"]), "limite del protocollo / comportamento del modello")
    # Decisione di Simona (14 nov, prima dei dati del completo): la baseline «p(UNKNOWN) normalizzata» resta nel
    # confronto, ma non è una baseline indipendente: normalizzata = p(UNKNOWN) / (p(YES)+p(NO)+p(UNKNOWN)). Se la massa
    # sulle tre risposte è ≈ 1 coincide con p(UNKNOWN), quindi una parità con essa NON è un'anomalia del metodo.
    # Si segnala solo se ne differisce (|ΔW| ≥ 0.02): allora conta la massa fuori dalle tre risposte.
    norm = base[NORMALIZZATA]["W"]["stima"]
    if norm is not None and w["value"] is not None and abs(norm - w["value"]) >= 0.02:
        _segnala(anomalie, "p(UNKNOWN) normalizzata dà un W diverso da p(UNKNOWN): la massa fuori da YES/NO/UNKNOWN conta",
                 f"{norm:.3f} contro {w['value']:.3f}; quota righe con massa < {MASSA_BASSA}: {e['quota_massa_bassa']:.1%}",
                 "comportamento del modello / problema tecnico (template, formato della risposta)")
    migliori = {k: v["W"]["stima"] for k, v in base.items()
                if k not in ("p_UNKNOWN (principale)", NORMALIZZATA) and not k.startswith("lunghezza")
                and v["W"]["stima"] is not None}
    if migliori and w["value"] is not None and max(migliori.values()) >= w["value"]:
        k = max(migliori, key=migliori.get)
        _segnala(anomalie, "una baseline indipendente dalle uscite eguaglia o supera p(UNKNOWN)",
                 f"{k}: {migliori[k]:.3f} contro {w['value']:.3f}", "evidenza scientifica")
    al = e.get("allineamento_test_U2")
    if al and not al.get("ambiente_allineato", True):
        _segnala(anomalie, "Qwen3-8B non riproduce il Test U2", json.dumps(al), "problema tecnico (ambiente, versioni)")
    lung = base["lunghezza del prompt in token"]["W"]["stima"]
    return {
        "1. Funziona tecnicamente?": dict(
            revisione_verificata=True, template_e_etichette="controllati", coppie_valide=w["n_pairs"],
            coppie_scartate_lunghezza=e["coppie_scartate_lunghezza"],
            esclusioni_per_attributo=dist["per_attributo"], W_con_coppie_escluse=sens,
            controlli_tokenizer=e.get("controlli_tokenizer"), risposte_ALTRO=risposte["ALTRO"],
            quota_massa_bassa=e["quota_massa_bassa"], controllo_positivo_Qwen=al),
        "2. Misura ciò che intendiamo?": dict(
            soglia_competenza=dict(superata=g["passed"], acc_B=g["accuracy_B"], per_gold=g["accuracy_B_by_gold"]),
            W_per_famiglia=rep["W_by_family_descriptive"],
            W_per_variante={k: v["stima"] for k, v in ctrl["W per variante"].items()},
            W_per_posizione={k: v["stima"] for k, v in ctrl["W per posizione"].items()},
            W_a_parita_di_risposta={k: v["stima"] for k, v in ctrl["W a parità di risposta generata"].items()},
            W_lunghezza_prompt=lung),
        "3. Aggiunge rispetto alle baseline?": dict(
            W_p_UNKNOWN=w["value"], IC95=[w["ci_lower"], w["ci_upper"]],
            W_baseline={k: v["W"]["stima"] for k, v in base.items() if not k.startswith("p_UNKNOWN (")},
            nota=("confronto descrittivo; non c'è un test preregistrato di superiorità sulle baseline per i modelli nuovi. "
                  + NOTA_NORMALIZZATA)),
        "anomalie": anomalie,
    }
print("Funzioni di analisi pronte.")

In [ ]:
# 4b. Controllo su CPU (obbligatorio prima del completo): l'analisi di questo notebook, applicata ai punteggi
# archiviati del Test U2, deve ridare esattamente i 4 W pubblicati su PN e i conteggi della soglia di competenza.
def controllo_cpu_U2():
    import io
    import pandas as pd
    percorso = Path("/content/drive/MyDrive") / ARCHIVIO_TEST_U2 if USA_DRIVE else Path(ARCHIVIO_TEST_U2)
    if not percorso.is_file():
        return dict(eseguito=False, superato=False, motivo=f"archivio non trovato: {percorso}")
    with zipfile.ZipFile(percorso) as z:
        dati = z.read("punteggi.csv")
    sha = hashlib.sha256(dati).hexdigest()
    if sha != PUNTEGGI_U2_SHA256:
        return dict(eseguito=False, superato=False, motivo=f"punteggi.csv diverso dall'archiviato (SHA-256 {sha[:12]})")
    t = pd.read_csv(io.BytesIO(dati), keep_default_na=False)
    t = t[(t.set == "PN") & t.kind.isin(["B", "C"])]
    out = {}
    for key, d in t.groupby("modello"):
        rows = [dict(row_id=r.row_id, pair_id="PN|" + r.pair_key, block_id=r.story_id, side=r.kind,
                     score=r.p_UNKNOWN, p_yes=r.p_YES, p_no=r.p_NO, p_unknown=r.p_UNKNOWN, mass_total=r.mass_total,
                     generated=r.generated, parsed=r.parsed, prompt_tokens=r.prompt_tokens,
                     model_id=NOMI_U2[key], model_revision="archivio Test U2") for r in d.itertuples()]
        assert {r["row_id"] for r in rows} == {r["row_id"] for r in TUTTE}, "righe diverse dal benchmark congelato"
        tenute, scarti = filtro_lunghezza(rows, {r["row_id"]: r["prompt_tokens"] for r in rows})
        e = analizza(key, dict(id=NOMI_U2[key], sha="", ruolo="archivio"), tenute, scarti, False, tutte=rows)
        g = e["report_indetbench"]["competence_gate"]
        conteggi = {lab: (round(g["accuracy_B_by_gold"][lab] * g["n_B_by_gold"][lab]), g["n_B_by_gold"][lab])
                    for lab in ("NO", "YES")}
        w = e["report_indetbench"]["W_p_unknown"]
        ok = (abs(w["value"] - W_ATTESI_U2_PN[key]) < 1e-12 and w["n_pairs"] == 800 and
              conteggi == {k: tuple(v) for k, v in ob.REFERENCE_B_ACCURACY_BY_GOLD_PS[NOMI_U2[key]].items()} and
              e["esito"] == "CONFIRMED")
        out[key] = dict(W=w["value"], IC95=[w["ci_lower"], w["ci_upper"]], W_atteso=W_ATTESI_U2_PN[key],
                        conteggi_B_per_gold=conteggi, esito=e["esito"], superato=ok)
        print(f"  {key}: W {w['value']:.5f} (atteso {W_ATTESI_U2_PN[key]}) | {e['esito']} | "
              f"B corretti per gold {conteggi} | {'OK' if ok else 'DIVERSO'}")
    return dict(eseguito=True, superato=len(out) == 4 and all(v["superato"] for v in out.values()),
                sha256_punteggi=sha, modelli=out, repo_commit=REPO_COMMIT, versione=VERSIONE)

CONTROLLO_CPU = controllo_cpu_U2()
write_json(ROOT / "controllo_cpu_testU2.json", CONTROLLO_CPU)
print("Controllo CPU dei 4 W del Test U2:", "SUPERATO" if CONTROLLO_CPU["superato"] else
      f"NON SUPERATO ({CONTROLLO_CPU.get('motivo', 'valori diversi')})")
if not CONTROLLO_CPU["superato"] and not SMOKE:
    print("Il completo non partirà finché questo controllo non è superato.")

In [ ]:
# 5. Funzioni per la GPU: caricamento alla revisione fissata, controlli del tokenizer e del template, inferenza con checkpoint.
# L'inferenza usa indetbench.hf_runner.score_pairs al commit fissato (stesso calcolo del Test U2: probabilità del primo
# token, somma delle 6 varianti di scrittura per etichetta, nessuna rinormalizzazione, risposta greedy di 8 token).
import csv, gc, platform

BLOCCO = 50   # righe per file di checkpoint

def hf_token():
    return segreto("HF_TOKEN")

def famiglia_gpu():
    import torch
    nome = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return nome, ("A100" if "A100" in nome else nome)

def ambiente():
    import torch, transformers
    nome, fam = famiglia_gpu()
    return dict(python=platform.python_version(), torch=str(torch.__version__),
                transformers=transformers.__version__, gpu=nome, famiglia_gpu=fam)

def verifica_revisione(cfg, token):
    from huggingface_hub import HfApi
    info = HfApi().model_info(cfg["id"], revision=cfg["sha"], token=token)
    if info.sha != cfg["sha"]:
        raise RuntimeError(f"{cfg['id']}: la revisione {cfg['sha']} non corrisponde ({info.sha})")

def carica_tokenizer(cfg, token):
    return solo_ids_e_maschera(carica_tokenizer_originale(cfg, token))

def carica_tokenizer_originale(cfg, token):
    from transformers import AutoTokenizer
    extra = {"fix_mistral_regex": True} if cfg["key"] == "mistral" else {}
    tok = AutoTokenizer.from_pretrained(cfg["id"], revision=cfg["sha"], token=token, trust_remote_code=False, **extra)
    tok.padding_side = "left"
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    return tok

def solo_ids_e_maschera(tok):
    # Alcuni tokenizer restituiscono anche token_type_ids, che generate() rifiuta per i modelli causali.
    # Si tengono solo input_ids e attention_mask (come nei tokenizer dei 4 modelli originali). Adattamento solo
    # tecnico: controlli_tokenizer verifica su tutti i prompt che input_ids e attention_mask restino identici.
    tok.model_input_names = ["input_ids", "attention_mask"]
    return tok

def carica_modello(cfg, token):
    import torch, transformers
    from transformers import AutoModelForCausalLM
    ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    dtype_arg = "dtype" if ver >= (4, 56) else "torch_dtype"
    extra = {}
    if cfg.get("quant") == "nf4":
        from transformers import BitsAndBytesConfig
        extra["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                          bnb_4bit_compute_dtype=torch.bfloat16,
                                                          bnb_4bit_use_double_quant=False)
    return AutoModelForCausalLM.from_pretrained(
        cfg["id"], revision=cfg["sha"], token=token, device_map={"": 0}, attn_implementation="eager",
        trust_remote_code=False, use_safetensors=True, low_cpu_mem_usage=True,
        **{dtype_arg: torch.bfloat16}, **extra).eval()

def versione_bnb():
    try:
        import bitsandbytes
        return bitsandbytes.__version__
    except Exception:
        return "non installato"

def libera_cache(cfg):
    """Cancella dalla cache locale di Colab (non da Drive) i pesi del modello appena usato: i 5 modelli insieme
    superano il disco della sessione. Un rilancio li riscarica alla stessa revisione."""
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir()
        hashes = [rev.commit_hash for repo in info.repos if repo.repo_id == cfg["id"] for rev in repo.revisions]
        if hashes:
            info.delete_revisions(*hashes).execute()
            print(f"  cache locale di {cfg['id']} cancellata")
    except Exception as err:
        print("  ATTENZIONE: cache non cancellata:", nascondi(err)[:200])

def memoria_gpu_gb():
    import torch
    return torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0

def copertura_etichette(model, tok, cfg, ids, rows):
    """Il token più probabile al primo passo è fra le etichette misurate? Massa sulle etichette (controllo dopo Yi)."""
    import torch
    tutti = {i for v in ids.values() for i in v}
    argmax, masse = Counter(), []
    dentro = 0
    for r in rows:
        enc = tok(hr.chat_prompt(tok, r["prompt"], model_key=cfg["key"]), return_tensors="pt",
                  add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            p = model(**enc).logits[0, -1].float().softmax(-1)
        top = int(p.argmax())
        dentro += top in tutti
        argmax[tok.convert_ids_to_tokens(top)] += 1
        masse.append(float(p[sorted(tutti)].sum()))
    masse.sort()
    mediana = masse[len(masse) // 2]
    return dict(righe=len(rows), quota_token_piu_probabile_fra_etichette=dentro / len(rows),
                token_piu_probabili=dict(argmax.most_common(6)), massa_mediana=mediana,
                etichette_token={k: tok.convert_ids_to_tokens(v) for k, v in ids.items()},
                superato=mediana >= MASSA_MEDIANA_MIN)

def controlla_template(tok, cfg):
    """Il prompt deve dipendere solo dal testo: niente template mancante, niente data del giorno."""
    t = tok.chat_template
    if isinstance(t, dict):
        t = t.get("default")
    if not t:
        return "il tokenizer non ha un chat template"
    if "strftime_now" in t:
        return "il chat template inserisce la data del giorno (il prompt cambierebbe con la data di lancio)"
    a = hr.chat_prompt(tok, "Question: test?", model_key=cfg["key"])
    b = hr.chat_prompt(tok, "Question: test?", model_key=cfg["key"])
    if a != b or "Question: test?" not in a:
        return "il chat template non è deterministico o non contiene il messaggio"
    if cfg["key"] != "qwen" and "<think>" in a:
        return "il chat template apre un blocco di ragionamento"
    return None

def controlli_tokenizer(tok, tok_orig, cfg, rows):
    """(a) togliere token_type_ids non cambia input_ids né attention_mask su nessun prompt;
    (b) il prompt congelato compare intatto nel testo passato al modello."""
    diversi, mancanti = [], []
    for r in rows:
        testo = hr.chat_prompt(tok, r["prompt"], model_key=cfg["key"])
        if testo != hr.chat_prompt(tok_orig, r["prompt"], model_key=cfg["key"]):
            diversi.append(r["row_id"])
            continue
        if r["prompt"] not in testo:
            mancanti.append(r["row_id"])
        a = tok(testo, add_special_tokens=False)
        b = tok_orig(testo, add_special_tokens=False)
        if a["input_ids"] != b["input_ids"] or a["attention_mask"] != b["attention_mask"]:
            diversi.append(r["row_id"])
    return dict(model_input_names_originali=list(tok_orig.model_input_names),
                token_type_ids_tolti="token_type_ids" in tok_orig.model_input_names,
                prompt_controllati=len(rows), tokenizzazione_diversa=diversi[:20], n_tokenizzazione_diversa=len(diversi),
                prompt_non_intatti=mancanti[:20], n_prompt_non_intatti=len(mancanti),
                superato=not diversi and not mancanti,
                sha256_testi_al_modello=hashlib.sha256("\n".join(
                    hr.chat_prompt(tok, r["prompt"], model_key=cfg["key"]) for r in rows).encode()).hexdigest())

def lunghezze(tok, cfg, rows):
    return {r["row_id"]: len(tok(hr.chat_prompt(tok, r["prompt"], model_key=cfg["key"]),
                                 add_special_tokens=False)["input_ids"]) for r in rows}

def inferenza(model, tok, cfg, rows):
    return list(hr.score_pairs(model, tok, rows, model_key=cfg["key"], model_id=cfg["id"],
                               revision=cfg["sha"], max_new_tokens=8))

def con_checkpoint(cartella, rows, calcola):
    cartella.mkdir(parents=True, exist_ok=True)
    out = []
    for i in range(0, len(rows), BLOCCO):
        parte_rows = rows[i:i + BLOCCO]
        attesi = [r["row_id"] for r in parte_rows]
        path = cartella / f"{i // BLOCCO:04d}.json"
        if path.exists():
            parte = json.loads(path.read_text())
        else:
            parte = calcola(parte_rows)
            write_json(path, parte)
        if [r["row_id"] for r in parte] != attesi:
            raise RuntimeError(f"Checkpoint incoerente: {path}")
        out.extend(parte)
        print(f"  {len(out)}/{len(rows)}", flush=True)
    return out

def scrivi_csv(path, rows):
    tmp = path.with_suffix(".csv.tmp")
    campi = list(dict.fromkeys(k for r in rows for k in r)) or ["row_id"]
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campi, restval="")
        w.writeheader()
        w.writerows(rows)
    os.replace(tmp, path)

def esporta(cartella, esito, tabelle):
    write_json(cartella / "esito.json", esito)
    nomi = ["esito.json"]
    for nome, rows in tabelle.items():
        scrivi_csv(cartella / f"{nome}.csv", rows)
        nomi.append(f"{nome}.csv")
    archivio = cartella / f"{cartella.name}.zip"
    tmp = archivio.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for n in nomi:
            z.write(cartella / n, n)
    os.replace(tmp, archivio)
    return archivio

def valuta_modello(nome, token):
    import torch
    cfg = MODEL_CONFIG[nome]
    if memoria_gpu_gb() < cfg["gpu_min_gb"]:   # errore, non esito: rilanciando su una GPU adatta il modello riparte
        raise RuntimeError(f"{nome}: servono almeno {cfg['gpu_min_gb']} GB di memoria GPU, "
                           f"questa ne ha {memoria_gpu_gb():.0f} (usa la sessione A100-SXM4-80GB)")
    import shutil
    from huggingface_hub.constants import HF_HUB_CACHE
    Path(HF_HUB_CACHE).mkdir(parents=True, exist_ok=True)
    libero = shutil.disk_usage(HF_HUB_CACHE).free / 1e9
    gia = any(Path(HF_HUB_CACHE).glob(f"models--{cfg['id'].replace('/', '--')}/snapshots/{cfg['sha']}"))
    if not gia and libero < cfg["disco_gb"] + 5:
        raise RuntimeError(f"{nome}: servono circa {cfg['disco_gb']} GB di disco per i pesi, liberi {libero:.0f} GB")
    pesi = "bfloat16" if not cfg.get("quant") else f"{cfg['quant']} 4 bit (bitsandbytes {versione_bnb()}), calcolo bfloat16"
    manifest = dict(versione=VERSIONE, modello=cfg["id"], revisione=cfg["sha"], dtype=pesi, attention="eager",
                    batch=1, max_new_tokens=8, repo_commit=REPO_COMMIT, benchmark_sha256=ob.FROZEN_SHA256,
                    dati=DATA_SHA, modalita=MODALITA, **ambiente())
    impronta = dict(manifest, gpu=None)           # A100 40 GB e 80 GB equivalenti, come nel Test M
    cartella = ROOT / f"{nome}-{fingerprint(impronta)[:12]}"
    cartella.mkdir(parents=True, exist_ok=True)
    if (cartella / f"{cartella.name}.zip").is_file():
        print(f"{nome}: già completato ({cartella.name})")
        return cartella
    write_json(cartella / "manifest.json", manifest)
    def fermo(motivo, **extra):
        esito = dict(manifest, chiave=nome, ruolo=cfg["ruolo"], esito=f"NON MISURABILE: {motivo}", **extra)
        print(f"  {nome}: NON MISURABILE, {motivo}")
        esporta(cartella, esito, {})
        return cartella
    verifica_revisione(cfg, token)
    tok_orig = carica_tokenizer_originale(cfg, token)
    tok = carica_tokenizer(cfg, token)
    motivo = controlla_template(tok, cfg)
    if motivo:
        return fermo(motivo)
    try:
        ids = hr.token_label_ids(tok)
    except ValueError as err:   # es. YES/NO/UNKNOWN con lo stesso primo token
        return fermo(f"etichette non separabili nel tokenizer ({err})")
    righe_controllo = []
    if nome == "qwen_controllo":
        righe_controllo = [r for r in TUTTE if r["row_id"] in {x[0] for x in REF_QWEN_PN}]
    ctok = controlli_tokenizer(tok, tok_orig, cfg, RIGHE)
    print(f"  {nome}: token_type_ids tolti: {ctok['token_type_ids_tolti']} | tokenizzazione diversa "
          f"{ctok['n_tokenizzazione_diversa']} | prompt non intatti {ctok['n_prompt_non_intatti']}")
    if not ctok["superato"]:
        return fermo("controllo del tokenizer non superato", controlli_tokenizer=ctok)
    lung = lunghezze(tok, cfg, RIGHE)
    _, scarti = filtro_lunghezza(RIGHE, lung)
    print(f"  {nome}: etichette {ids} | coppie escluse per lunghezza {len({s['pair_id'] for s in scarti})} "
          "(misurate comunque, fuori dall'analisi principale)")
    model = carica_modello(cfg, token)
    try:
        extra = dict(label_ids=ids, manifest=manifest, controlli_tokenizer=ctok)
        controllo_etich = [r for r in TUTTE if r["row_id"] in {x[0] for x in REF_QWEN_PN}]   # V2001–V2003, 24 righe
        cop = copertura_etichette(model, tok, cfg, ids, controllo_etich)
        extra["copertura_etichette"] = cop
        print(f"  {nome}: massa mediana sulle etichette {cop['massa_mediana']:.3g}, token più probabile fra le "
              f"etichette {cop['quota_token_piu_probabile_fra_etichette']:.0%}, token più probabili {cop['token_piu_probabili']}")
        if not cop["superato"]:
            return fermo(f"massa mediana sulle etichette {cop['massa_mediana']:.3g} < {MASSA_MEDIANA_MIN}: le etichette "
                         "misurate non coprono il token che il modello genera", copertura_etichette=cop,
                         controlli_tokenizer=ctok)
        if righe_controllo:
            ref = {r: (v, p) for r, v, p in REF_QWEN_PN}
            out = inferenza(model, tok, cfg, righe_controllo)
            delta = max(abs(float(o["score"]) - ref[o["row_id"]][0]) for o in out)
            uguali = sum(o["parsed"] == ref[o["row_id"]][1] for o in out)
            extra["allineamento_test_U2"] = dict(n=len(out), scarto_max=delta, superato=delta <= TOLL_VALORI,
                                                 risposte_uguali=uguali)
            print(f"  Controllo Qwen: scarto max {delta:.3g} su {len(out)} valori (soglia {TOLL_VALORI}), "
                  f"risposte uguali {uguali}/{len(out)}")
        raw_tutte = con_checkpoint(cartella / f"checkpoint-{DATA_SHA[:12]}", RIGHE,
                                   lambda rows: inferenza(model, tok, cfg, rows))
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        libera_cache(cfg)
    for o in raw_tutte:   # la lunghezza usata nell'inferenza deve essere quella del controllo
        if int(o["prompt_tokens"]) != lung[o["row_id"]]:
            raise RuntimeError(f"Lunghezza diversa fra controllo e inferenza: {o['row_id']}")
    escluse = {s["pair_id"] for s in scarti}
    raw = [o for o in raw_tutte if o["pair_id"] not in escluse]
    if not raw:
        esporta(cartella, dict(manifest, chiave=nome, ruolo=cfg["ruolo"], controlli_tokenizer=ctok,
                               esito="NON MISURABILE: nessuna coppia con B e C della stessa lunghezza"),
                dict(punteggi=raw_tutte, scarti_lunghezza=scarti))
        print(f"  {nome}: NON MISURABILE, tutte le coppie escluse per lunghezza")
        return cartella
    extra["identita"] = dict(dati_sha256=DATA_SHA, benchmark_sha256=ob.FROZEN_SHA256,
                             testi_al_modello_sha256=ctok["sha256_testi_al_modello"],
                             coppie_valide_sha256=fingerprint(sorted({o["pair_id"] for o in raw})),
                             n_coppie_valide=len({o["pair_id"] for o in raw}))
    esito = analizza(nome, cfg, raw, scarti, SMOKE, extra, tutte=raw_tutte)
    masse = sorted(float(o["mass_total"]) for o in raw)
    esito["massa_mediana_tutte"] = masse[len(masse) // 2]
    if esito["massa_mediana_tutte"] < MASSA_MEDIANA_MIN:
        esito["esito_del_codice_prima_della_regola_massa"] = esito["esito"]
        esito["esito"] = (f"NON MISURABILE: massa mediana sulle etichette {esito['massa_mediana_tutte']:.3g} "
                          f"< {MASSA_MEDIANA_MIN} (regola tecnica del 15 nov)")
    if nome == "qwen_controllo":
        a = esito["allineamento_test_U2"]
        if not SMOKE:
            w = esito["report_indetbench"]["W_p_unknown"]["value"]
            a["W_PN"] = dict(stima=w, test_U2=W_QWEN_PN_TEST_U2, superato=abs(w - W_QWEN_PN_TEST_U2) <= TOLL_W)
        a["ambiente_allineato"] = a["superato"] and a.get("W_PN", {}).get("superato", True)
    esito["livelli"] = livelli(esito)
    archivio = esporta(cartella, esito, dict(punteggi=[dict(o, coppia_esclusa_lunghezza=o["pair_id"] in escluse)
                                                       for o in raw_tutte], scarti_lunghezza=scarti,
                                             anomalie=esito["livelli"]["anomalie"]))
    print(f"  {nome}: {esito['esito']} | salvato {archivio.name}")
    return cartella
print("Funzioni GPU definite (nessun modello caricato).")

In [ ]:
# 6. Avvio esplicito. Se la sessione cade, rilancia tutte le celle: i blocchi già salvati su Drive non si ricalcolano.
def riepilogo_totale(cartelle):
    esiti = {}
    for nome, c in cartelle.items():
        esiti[nome] = json.loads((c / "esito.json").read_text())
    allineato = esiti.get("qwen_controllo", {}).get("allineamento_test_U2", {}).get("ambiente_allineato")
    identita = identita_coppie(esiti, cartelle)
    righe, note = [], []
    for nome, e in esiti.items():
        if "report_indetbench" in e:
            righe.append(riga_riepilogo(e))
        else:
            righe.append(dict(modello=nome, esito=e["esito"]))
    if allineato is None:
        note.append("Controllo positivo Qwen non eseguito in questa sessione: confronto con il Test U2 non verificato.")
    elif not allineato:
        note.append("ATTENZIONE: Qwen3-8B non riproduce il Test U2 entro le tolleranze; i valori dei modelli nuovi "
                    "non sono confrontabili con i resoconti originali finché la causa non è chiarita.")
    if not identita["stessi_prompt"]:
        note.append("ATTENZIONE: i modelli non hanno ricevuto gli stessi prompt congelati (impronte diverse).")
    if not CONTROLLO_CPU["superato"]:
        note.append("Controllo CPU dei 4 W del Test U2 non superato: " + CONTROLLO_CPU.get("motivo", "valori diversi"))
    note += ["I risultati dei modelli nuovi sono ESPLORATIVI: non sono una validazione indipendente del metodo.",
             "Un W alto non dimostra da solo la validità del metodo: leggerlo insieme alle baseline dalle uscite "
             "(entropia, meno p massima, risposta UNKNOWN) e ai controlli per famiglia, variante e posizione.",
             NOTA_NORMALIZZATA,
             "Il W della lunghezza del prompt vale 0.5 per costruzione: le coppie con B e C di lunghezza diversa sono escluse."]
    if SMOKE:
        note.insert(0, "MODALITÀ PROVA: verifica tecnica, nessuna lettura scientifica.")
    return esiti, righe, note, identita

def identita_coppie(esiti, cartelle):
    """Stesse coppie B/C per tutti i modelli: stessi prompt congelati; coppie valide comuni e W su di esse."""
    import pandas as pd
    misurati = {k: e for k, e in esiti.items() if "identita" in e}
    impronte = {k: (e["identita"]["dati_sha256"], e["identita"]["benchmark_sha256"]) for k, e in misurati.items()}
    valide = {}
    for k in misurati:
        t = pd.read_csv(cartelle[k] / "punteggi.csv", keep_default_na=False)
        valide[k] = set(t.loc[t.coppia_esclusa_lunghezza.astype(str) != "True", "pair_id"])
    comuni = set.intersection(*valide.values()) if valide else set()
    W_comuni = {}
    for k in misurati:
        t = pd.read_csv(cartelle[k] / "punteggi.csv", keep_default_na=False).to_dict("records")
        W_comuni[k] = stima_W(t, MISURE["p_UNKNOWN (principale)"], 0 if SMOKE else 2000, comuni)
    return dict(stessi_prompt=len(set(impronte.values())) <= 1, impronte=impronte,
                coppie_valide={k: len(v) for k, v in valide.items()}, coppie_comuni=len(comuni),
                W_su_coppie_comuni_descrittivo=W_comuni)

if AVVIA_GPU:
    if not SMOKE and not CONTROLLO_CPU["superato"]:
        raise RuntimeError("Completo fermato: il controllo CPU dei 4 W del Test U2 (cella 4b) non è superato.")
    nome_gpu, fam = famiglia_gpu()
    print("GPU:", nome_gpu)
    if SOLO_A100 and fam != "A100":
        raise RuntimeError(f"GPU {nome_gpu}: serve una A100 (SOLO_A100 = True)")
    if any(MODEL_CONFIG[k].get("quant") for k in MODELLI) and versione_bnb() == "non installato":
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError("Installazione di bitsandbytes fallita: " + r.stderr[-300:])
        print("bitsandbytes installato:", versione_bnb(), "(versione registrata nel manifest del modello quantizzato)")
    print(f"Memoria GPU: {memoria_gpu_gb():.0f} GB")
    token = hf_token()
    if token is None:
        print("ATTENZIONE: HF_TOKEN non disponibile; Gemma (gated) non si scaricherà.")
    ordine = sorted(MODELLI, key=lambda k: k != "qwen_controllo")   # il controllo positivo per primo
    cartelle, errori = {}, {}
    for nome in ordine:
        print(f"\n=== {nome}: {MODEL_CONFIG[nome]['id']} @ {MODEL_CONFIG[nome]['sha'][:7]} ===")
        try:
            cartelle[nome] = valuta_modello(nome, token)
        except Exception as err:    # un modello che fallisce non ferma gli altri; l'errore resta nel rapporto
            errori[nome] = nascondi(f"{type(err).__name__}: {err}")[:500]
            print(f"  ERRORE {nome}: {errori[nome]}")
    esiti, righe, note, identita = riepilogo_totale(cartelle)
    righe += [dict(modello=k, esito=f"ERRORE: {v}") for k, v in errori.items()]
    write_json(ROOT / "riepilogo.json", dict(versione=VERSIONE, modalita=MODALITA, repo_commit=REPO_COMMIT,
                                              note=note, righe=righe, errori=errori, identita_coppie=identita,
                                              controllo_cpu_testU2=CONTROLLO_CPU))
    scrivi_csv(ROOT / "riepilogo.csv", righe)
    parti = [f"IndetBench, modelli nuovi | {VERSIONE} | modalità {MODALITA} | repository {REPO_COMMIT[:7]}", ""] + note
    for nome, e in esiti.items():
        parti += ["", f"===== {nome} ({e.get('modello')}) | {e.get('classe', '')} | esito: {e['esito']} ====="]
        for livello, contenuto in e.get("livelli", {}).items():
            if livello == "anomalie":
                parti.append(f"Anomalie da classificare ({len(contenuto)}): comportamento del modello, problema tecnico, "
                             "limite del protocollo o evidenza scientifica; per ognuna causa ipotizzata, prove, controlli, incertezze.")
                parti += [f"  - {a['anomalia']} [{a['prova']}] possibili: {a['categorie_possibili']}" for a in contenuto]
            else:
                parti.append(f"{livello} {json.dumps(contenuto, ensure_ascii=False)}")
    parti += ["", "Identità delle coppie fra i modelli: " + json.dumps(identita, ensure_ascii=False)]
    parti += ["", "Errori:"] + [f"  {k}: {v}" for k, v in errori.items()] if errori else []
    testo = "\n".join(parti)
    (ROOT / "rapporto.txt").write_text(testo, encoding="utf-8")
    finale = ROOT / f"indetbench_modelli_grandi_{MODALITA}.zip"
    with zipfile.ZipFile(finale.with_suffix(".tmp"), "w", zipfile.ZIP_DEFLATED) as z:
        for n in ("riepilogo.json", "riepilogo.csv", "rapporto.txt", "controllo_cpu_testU2.json"):
            z.write(ROOT / n, n)
        for nome, c in cartelle.items():
            for f in sorted(c.glob("*.json")) + sorted(c.glob("*.csv")):
                z.write(f, f"{c.name}/{f.name}")
    os.replace(finale.with_suffix(".tmp"), finale)
    print("\n" + testo)
    print(f"\nZIP: {finale}")
else:
    print("Preflight concluso. Imposta AVVIA_GPU = True per eseguire la modalità selezionata.")